## 🎯 Learning Objectives
* Understand the critical role of tracing and evaluation in production AI agent systems.
* Learn to integrate LangSmith for real-time, end-to-end tracing of LangChain agents.
* Explore how to leverage LangSmith for creating evaluation datasets and assessing agent performance.
* Identify best practices for deploying and monitoring LangChain agents with LangSmith in a production environment.


## LangSmith: Your Agent's Black Box Recorder and Performance Coach

Imagine you're building a complex autonomous vehicle. Would you launch it without a 'black box' recorder to log every decision, sensor reading, and action? And would you improve it without rigorous testing and performance metrics? Absolutely not. The same logic applies to AI agents in production.

As we move into 2026, AI agents are becoming increasingly sophisticated, handling critical tasks, and interacting with complex environments. Their non-deterministic nature, multi-step reasoning, and reliance on external tools make them incredibly powerful but also notoriously difficult to debug, understand, and improve. Traditional logging falls short; you need a holistic view of *what* your agent did, *why* it did it, and *how* well it performed.

This is where **LangSmith** comes in. LangSmith is an observability and evaluation platform specifically designed for large language model (LLM) applications, especially those built with LangChain. Think of it as:

1.  **The Flight Recorder (Tracing):** LangSmith provides end-to-end visibility into every step of your agent's execution. It visualizes the entire chain of operations – every LLM call, every tool invocation, every intermediate thought process. This 'trace' is like a detailed forensic report, invaluable for debugging, understanding unexpected behavior, and gaining insights into your agent's decision-making.

2.  **The Performance Coach (Evaluation):** Beyond just seeing what happened, LangSmith allows you to systematically measure *how well* your agent performed. You can create datasets of inputs and expected outputs, run your agent against them, and use automated or human-in-the-loop evaluators to score its performance. This is crucial for:
    *   **Iterative Improvement:** Identify weaknesses, refine prompts, improve tool descriptions, or even fine-tune underlying LLMs.
    *   **Regression Testing:** Ensure new changes or model updates don't degrade existing performance.
    *   **Benchmarking:** Compare different agent architectures or LLM providers.

### Why is LangSmith Essential for 2026 Production Agents?

*   **Complexity Management:** Modern agents are not simple prompt-response systems. They involve intricate orchestration, tool use, memory, and multi-turn conversations. LangSmith untangles this complexity.
*   **Reliability & Trust:** For agents to be trusted in production, their behavior must be understandable and auditable. LangSmith provides that audit trail.
*   **Rapid Iteration:** The agent development lifecycle is highly iterative. LangSmith accelerates this by providing fast feedback loops on changes.
*   **Cost Optimization:** By understanding exactly how your agent uses LLMs and tools, you can optimize token usage and API calls, leading to significant cost savings.

### How it Works (High-Level Steps):

1.  **Instrumentation:** Set a few environment variables (`LANGCHAIN_TRACING_V2`, `LANGCHAIN_API_KEY`, `LANGCHAIN_PROJECT`) in your LangChain application. LangChain automatically sends trace data to LangSmith.
2.  **Execution & Tracing:** Run your agent as usual. Every component's execution (LLM calls, tool calls, chain runs) is captured and sent to the LangSmith UI.
3.  **Analysis:** Navigate to the LangSmith UI to view detailed traces, identify bottlenecks, and debug issues.
4.  **Dataset Creation:** Define a set of inputs and corresponding expected outputs (ground truth) for your agent's tasks.
5.  **Evaluation Runs:** Execute your agent against the dataset. LangSmith records the agent's outputs and applies evaluators (either built-in or custom) to score performance.
6.  **Performance Review:** Analyze evaluation results in the LangSmith UI, compare different agent versions, and pinpoint areas for improvement.


In [ ]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.agents import AgentExecutor, create_openai_functions_agent
from langchain import hub
from langchain.tools import tool
from langsmith import Client
from langsmith.evaluation import evaluate

# Load environment variables from a .env file
# Ensure you have LANGCHAIN_API_KEY and OPENAI_API_KEY set in your .env file
# Example .env content:
# OPENAI_API_KEY="sk-..."
# LANGCHAIN_API_KEY="ls__..."
load_dotenv()

# --- Configure LangSmith for Tracing ---
# Enable LangSmith tracing (V2 is the modern standard)
os.environ["LANGCHAIN_TRACING_V2"] = "true"
# Set your LangSmith API key
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY")
# Assign a project name. All runs under this name will appear together in the LangSmith UI.
os.environ["LANGCHAIN_PROJECT"] = "AG04-L09-LangSmith-Tracing-Evals-2026"

# Ensure OpenAI API key is set for the LLM
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

# --- 1. Define a simple tool for our agent ---
@tool
def multiply(a: int, b: int) -> int:
    """Multiplies two integers together."""
    return a * b

@tool
def add(a: int, b: int) -> int:
    """Adds two integers together."""
    return a + b

tools = [multiply, add]

# --- 2. Create a simple LangChain agent ---
# Using a powerful model like gpt-4o for better agent reasoning
llm = ChatOpenAI(model="gpt-4o", temperature=0)

# Pull a standard prompt for OpenAI functions agent from LangChain Hub
prompt = hub.pull("hwchase17/openai-functions-agent")

# Create the agent
agent = create_openai_functions_agent(llm, tools, prompt)

# Create the AgentExecutor to run the agent
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

print("--- Running Agent with LangSmith Tracing Enabled ---")
try:
    # When this agent_executor.invoke() runs, its trace will be sent to LangSmith
    # You can find it under the project name specified in LANGCHAIN_PROJECT
    result = agent_executor.invoke({"input": "What is 123 multiplied by 456, then add 100?"})
    print(f"\nAgent Output: {result['output']}")
    print(f"\nCheck LangSmith UI for traces under project: {os.getenv('LANGCHAIN_PROJECT')}")
except Exception as e:
    print(f"\nError during agent invocation: {e}")
    print("Please ensure your LANGCHAIN_API_KEY and OPENAI_API_KEY are correctly set in your .env file.")

# --- 3. Demonstrate basic evaluation setup with LangSmith ---
print("\n--- Setting up LangSmith Evaluation ---")

# Initialize LangSmith client
client = Client()

# Define a dataset for evaluation
dataset_name = "AG04-L09 Math Agent Evaluation Dataset"
dataset_description = "Simple math problems for agent evaluation, demonstrating LangSmith evals."

# Check if dataset already exists to avoid recreation errors
try:
    dataset = client.read_dataset(dataset_name=dataset_name)
    print(f"Dataset '{dataset_name}' already exists. Using existing dataset.")
except Exception:
    print(f"Creating new dataset: '{dataset_name}'")
    dataset = client.create_dataset(
        dataset_name=dataset_name,
        description=dataset_description,
        input_keys=["input"],
        output_keys=["expected_output"],
    )
    client.create_example(
        dataset_id=dataset.id,
        inputs={"input": "What is 5 times 7?"},
        outputs={"expected_output": "35"},
    )
    client.create_example(
        dataset_id=dataset.id,
        inputs={"input": "Add 10 to 20."},
        outputs={"expected_output": "30"},
    )
    client.create_example(
        dataset_id=dataset.id,
        inputs={"input": "What is 100 minus 50?"}, # Agent doesn't have a subtract tool, good for testing failure
        outputs={"expected_output": "50"},
    )
    print(f"Dataset '{dataset_name}' created with examples.")

# Define a custom evaluation function.
# In a real-world scenario, you'd use more robust logic or LangSmith's built-in evaluators.
# This example checks if the agent's final output contains the expected numerical answer.
def custom_math_accuracy_evaluator(run, example) -> dict:
    """
    Custom evaluator to check if the agent's output contains the expected numerical answer.
    This is a very basic check and should be replaced with more robust logic for production.
    """
    agent_output = run.outputs.get("output", "") if run.outputs else ""
    expected_output = example.outputs.get("expected_output", "") if example.outputs else ""

    score = 0
    feedback = "Could not parse numbers or no match"

    try:
        # Attempt to extract numbers from both agent output and expected output
        # This is a simplified approach; real parsing might involve regex or more context.
        agent_num_str = "".join(filter(str.isdigit, agent_output))
        expected_num_str = "".join(filter(str.isdigit, expected_output))

        if agent_num_str and expected_num_str:
            agent_num = int(agent_num_str)
            expected_num = int(expected_num_str)
            if agent_num == expected_num:
                score = 1
                feedback = "Correct numerical answer found"
            else:
                feedback = f"Incorrect number. Expected {expected_num}, got {agent_num}"
        elif expected_num_str and expected_num_str in agent_output:
             # Fallback for cases where agent output might not be purely numeric but contains the answer
             score = 1
             feedback = "Expected number found in output string"
        else:
            feedback = "No clear numerical match"

    except ValueError:
        feedback = "Error parsing numbers for comparison"
    except Exception as e:
        feedback = f"Evaluator error: {e}"

    return {"score": score, "key": "custom_math_accuracy", "comment": feedback}

# Run the evaluation
print(f"Running evaluation on dataset '{dataset_name}'...")
try:
    evaluation_results = evaluate(
        lambda input_dict: agent_executor.invoke(input_dict), # The agent (or chain) to evaluate
        data=dataset_name, # The dataset to run against
        evaluators=[custom_math_accuracy_evaluator], # Our custom evaluator
        experiment_prefix="AG04-L09-Eval-Run", # A prefix for this evaluation run in LangSmith
        metadata={"lesson_id": "AG04-L09", "agent_version": "1.0"},
        max_concurrency=1, # Keep concurrency low for local testing/demonstration
        # You can also specify a project name for the evaluation results if different from tracing
        # project_name="AG04-L09-Evals-Project"
    )
    print(f"\nEvaluation completed. View results at: {evaluation_results['url']}")
except Exception as e:
    print(f"\nError during evaluation: {e}")
    print("Please ensure your LANGCHAIN_API_KEY is correctly set and the dataset exists/is created successfully.")


### Interpreting the Output and Leveraging LangSmith

After running the code, you'll see console output indicating the agent's response and, crucially, URLs for both the tracing project and the evaluation run in LangSmith. Let's break down what to look for:

#### 1. Tracing (`LANGCHAIN_PROJECT`)

When you navigate to the LangSmith UI (e.g., `https://smith.langchain.com/`) and select the project named `AG04-L09-LangSmith-Tracing-Evals-2026` (or whatever you set `LANGCHAIN_PROJECT` to), you'll find a list of "runs." Each `agent_executor.invoke()` call corresponds to a single run. Clicking on a run reveals a detailed waterfall diagram:

*   **Visual Flow:** You'll see the entire sequence of operations: the initial agent invocation, the LLM calls for reasoning, the tool invocations (e.g., `multiply`, `add`), and their respective outputs.
*   **Intermediate Steps:** For agents, you can observe the `Agent Thought` process, seeing exactly what the LLM decided to do at each step before calling a tool or formulating a final answer.
*   **Inputs & Outputs:** Every component's inputs (prompts, tool arguments) and outputs (LLM responses, tool results) are recorded.
*   **Timings & Costs:** LangSmith provides metrics on how long each step took and the token usage/estimated cost for LLM calls.

**How to use this:** This visual trace is your primary debugging tool. If your agent gives an incorrect answer, you can trace back to see if the LLM hallucinated, if a tool was called with wrong arguments, or if the tool's output was misinterpreted. It's like having a debugger for your agent's thought process.

#### 2. Evaluation (`evaluation_results['url']`)

The `evaluate` function generates a dedicated evaluation run in LangSmith. Clicking on the provided URL will take you to a dashboard showing:

*   **Overall Metrics:** You'll see aggregate scores based on the evaluators you defined (e.g., the average `custom_math_accuracy` score across all examples).
*   **Individual Example Results:** For each example in your dataset, you'll see:
    *   The input provided to the agent.
    *   The agent's actual output.
    *   The `expected_output` (ground truth).
    *   The score and feedback from your custom evaluator (e.g., "Correct numerical answer found" or "Incorrect number").
    *   Crucially, a link to the **trace** for that specific evaluation run. This allows you to deep-dive into *why* an agent failed on a particular example.

**How to use this:** Evaluation is key for iterative improvement. Identify examples where your agent performed poorly. Use the linked traces to understand the failure mode. Was it a prompt issue? A tool limitation? A reasoning error? Then, refine your agent, run the evaluation again, and compare the results to track progress.

### Performance Trade-offs and Considerations

*   **Overhead:** Enabling LangSmith tracing introduces a slight network overhead as data is sent to the LangSmith API. For most production applications, this is negligible compared to the benefits of observability. However, in extremely low-latency scenarios, you might consider selective tracing.
*   **Cost:** LangSmith has its own usage-based pricing model. Additionally, running extensive evaluations, especially with large datasets and powerful LLMs, will incur costs from your LLM provider. Plan your evaluation strategy wisely.
*   **Data Volume:** Tracing can generate a significant amount of data, especially for complex agents with many steps. LangSmith provides tools for managing projects and data retention, but it's something to be aware of.
*   **Privacy:** Ensure that any sensitive data processed by your agents is handled according to your organization's privacy policies when sent to LangSmith.

### Typical Use Cases in Production

*   **Debugging & Root Cause Analysis:** The most immediate benefit. Quickly pinpoint errors in complex agent workflows.
*   **Regression Testing:** Before deploying a new agent version, run it against a comprehensive evaluation dataset to ensure no existing functionality is broken.
*   **A/B Testing & Model Comparison:** Compare the performance of different LLMs, prompt strategies, or agent architectures side-by-side using evaluation metrics.
*   **Continuous Improvement:** Use evaluation results to systematically identify areas for improvement, leading to more robust and reliable agents over time.
*   **Monitoring Production Agents:** Integrate LangSmith into your production deployment to get real-time insights into agent behavior, performance, and potential failures in the wild.
*   **Human-in-the-Loop Feedback:** LangSmith supports human annotation and feedback, allowing you to gather qualitative insights and refine your ground truth datasets.

By integrating LangSmith, you transform agent development from an opaque, trial-and-error process into a data-driven, observable, and systematically improvable engineering discipline. This is crucial for building and maintaining production-grade AI agents in 2026 and beyond.


### Resources

*   **LangSmith Documentation:** The official and most comprehensive guide to all LangSmith features.
    *   [https://docs.smith.langchain.com/](https://docs.smith.langchain.com/)
*   **LangChain Tracing with LangSmith:** Detailed information on setting up and interpreting traces.
    *   [https://python.langchain.com/docs/langsmith/tracing](https://python.langchain.com/docs/langsmith/tracing)
*   **LangChain Evaluation with LangSmith:** Learn more about creating datasets, running evaluations, and using different evaluators.
    *   [https://python.langchain.com/docs/langsmith/evaluation](https://python.langchain.com/docs/langsmith/evaluation)
*   **LangSmith Hub:** Explore shared prompts, chains, and datasets.
    *   [https://smith.langchain.com/hub](https://smith.langchain.com/hub)
